# Basismodell ohne Adapter nach MLX/4 Bit konvertieren

Modell wählen, dann alle Zellen ausführen. Die Revisionen entsprechen dem Training. Das Ergebnis
liegt unter `omlx_ready/base_models/` in Drive und wird auf dem Mac nach `~/.omlx/models/` kopiert.

In [ ]:
# @title Modell und Backend
MODEL_ID = "Qwen/Qwen3-4B" # @param ["Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "microsoft/Phi-4-mini-instruct", "mistralai/Mistral-7B-Instruct-v0.3", "meta-llama/Llama-3.1-8B-Instruct"]
MLX_BACKEND = "cpu" # @param ["cpu", "cuda12", "cuda13"]

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'mlx[{MLX_BACKEND}]==0.32.3', 'mlx-lm==0.31.3',
                'transformers==5.15.1', 'safetensors==0.8.0'], check=True)

In [ ]:
import json, shutil
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from google.colab import drive
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/llm_phishing_finetune_workflow_v3')
BASE_MODELS = {
    'Qwen/Qwen3-4B': ('qwen3-4b-base-4bit', '1cfa9a7208912126459214e8b04321603b3df60c'),
    'Qwen/Qwen3-8B': ('qwen3-8b-base-4bit', 'b968826d9c46dd6066d109eabc6255188de91218'),
    'microsoft/Phi-4-mini-instruct': ('phi4-mini-instruct-base-4bit', 'cfbefacb99257ffa30c83adab238a50856ac3083'),
    'mistralai/Mistral-7B-Instruct-v0.3': ('mistral-7b-instruct-v0.3-base-4bit', 'c170c708c41dac9275d15a8fff4eca08d52bab71'),
    'meta-llama/Llama-3.1-8B-Instruct': ('llama31-8b-instruct-base-4bit', '0e9e39f249a16976918f6564b8830bc894c89659'),
}

def mlx_convert(source, output):
    """4-Bit-affine Quantisierung (Gruppengröße 64) eines lokalen HF-Ordners nach MLX."""
    import mlx.core as mx
    from mlx_lm import convert
    config = json.loads((source / 'config.json').read_text())
    if 'rope_theta' not in config and isinstance(config.get('rope_parameters'), dict):
        config['rope_theta'] = config['rope_parameters'].get('rope_theta')
        (source / 'config.json').write_text(json.dumps(config, indent=2))
    mx.set_default_device(mx.cpu if MLX_BACKEND == 'cpu' else mx.gpu)
    shutil.rmtree(output, ignore_errors=True)
    convert(str(source), mlx_path=str(output), quantize=True, q_bits=4, q_group_size=64, q_mode='affine', dtype='float16')
    for pattern in ('*.jinja', 'LICENSE*', 'NOTICE*'):
        for path in source.glob(pattern):
            if path.is_file() and not (output / path.name).exists():
                shutil.copy2(path, output / path.name)

def save_to_drive(output, destination, provenance, identity):
    (output / 'artifact_provenance.json').write_text(json.dumps(provenance, indent=2))
    files = {str(p.relative_to(output)): p.stat().st_size for p in sorted(output.rglob('*')) if p.is_file()}
    (output / 'conversion_manifest.json').write_text(json.dumps({
        'manifest_version': 2, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'identity': identity, 'device': MLX_BACKEND, 'files': files}, indent=2))
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output, destination)
    print('Modell in Drive gespeichert:', destination)

In [ ]:
name, revision = BASE_MODELS[MODEL_ID]
token = getpass('Hugging Face Read-Token: ').strip() if MODEL_ID.startswith('meta-llama/') else False
local_source, local_output = Path('/content/hf_source'), Path('/content/mlx_model')
snapshot_download(repo_id=MODEL_ID, revision=revision, local_dir=str(local_source), token=token,
                  allow_patterns=['*.json', 'model*.safetensors', '*.model', '*.txt', '*.jinja', '*.tiktoken', 'LICENSE*', 'NOTICE*'])
mlx_convert(local_source, local_output)
provenance = {'base_model_id': MODEL_ID, 'base_revision': revision, 'kind': 'unadapted', 'task': None,
              'adapter_applied': False, 'adapter_source': None, 'input_mode': 'text_only'}
save_to_drive(local_output, PROJECT / 'omlx_ready/base_models' / name, provenance,
              {**provenance, 'bits': 4, 'group_size': 64, 'mode': 'affine', 'dtype': 'float16'})